# Agent 流式输出与输出模式

> 展示 `create_agent` 的流式输出（`stream_mode`）及多种输出模式的用法。
>
> `create_agent` 返回一个 LangGraph `CompiledStateGraph`，支持标准 LangGraph 的 stream_mode。

## 0. 准备工作：定义工具与 Agent

> 工具定义 + Agent 工厂函数，避免重复代码。

In [1]:
import os
import dotenv
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from rich import print as rprint
from rich.panel import Panel

dotenv.load_dotenv()

@tool
def get_weather(city: str) -> str:
    """根据城市名称查询当前天气信息。"""
    mock_weather = {
        "北京": "晴天，25°C",
        "上海": "多云，22°C",
        "广州": "小雨，28°C",
        "深圳": "晴，30°C",
    }
    return mock_weather.get(city, f"未找到{city}的天气信息")

@tool
def calculate(expression: str) -> str:
    """计算数学表达式，如 '(3+5)*2'。"""
    return str(eval(expression))

@tool
def get_current_time() -> str:
    """获取当前时间。"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tools = [get_weather, calculate, get_current_time]

# Agent 工厂：简化重复创建
def make_agent(**kwargs):
    return create_agent(
        model="openai:" + os.getenv("MODEL_NAME"),
        tools=tools,
        system_prompt="你是一个有帮助的助手。",
        checkpointer=MemorySaver(),
        **kwargs,
    )

print("✅ 工具已加载，Agent 工厂已就绪")

✅ 工具已加载，Agent 工厂已就绪


---

# 第一部分：stream_mode 详解

> `stream_mode` 参数控制 `.stream()` / `.astream()` 以何种粒度产出事件。
>
> 常见模式：
> - `"updates"`（默认）— 每个节点执行完 emit 一次，输出该节点的增量变更
> - `"values"` — 每个节点执行完 emit 一次，输出完整状态快照
> - `"messages"` — 逐 token 发射 LLM 消息块，实现打字机效果

## 1. stream_mode="updates"（默认模式）

> 每步仅输出发生变化的节点数据。适合追踪「哪个节点刚执行完、输出了什么」。

In [ ]:
agent = make_agent()

print("=" * 60)
print("【stream_mode='updates'】 — 每次 emit 仅输出增量变更")
print("=" * 60)

for event in agent.stream(
    {"messages": [{"role": "user", "content": "北京天气怎么样？"}]},
    {"configurable": {"thread_id": "updates-001"}},
    stream_mode="updates",
):
    for node_name, update in event.items():
        print(f"\n🔶 节点 [{node_name}] 产生更新：")
        for msg in update.get("messages", []):
            role = msg.type  # Pydantic 对象用属性访问
            content = msg.content or ""
            tc = msg.tool_calls if hasattr(msg, "tool_calls") else []
            if tc:
                print(f"   🤖 AI 调用工具: {[(t['name'], t['args']) for t in tc]}")
            elif content:
                print(f"   [{role}]: {str(content)[:100]}")

## 2. stream_mode="values"

> 每步输出完整的 Agent 状态（包含所有历史消息）。适合需要「每一步的完整快照」的场景。

In [ ]:
agent = make_agent()

print("=" * 60)
print("【stream_mode='values'】 — 每次 emit 输出完整状态")
print("=" * 60)

for step, snapshot in enumerate(agent.stream(
    {"messages": [{"role": "user", "content": "上海天气怎么样？"}]},
    {"configurable": {"thread_id": "values-001"}},
    stream_mode="values",
)):
    msgs = snapshot.get("messages", [])
    print(f"\n📸 Step {step} — 共 {len(msgs)} 条消息：")
    for i, msg in enumerate(msgs):
        role = msg.type  # Pydantic 对象用属性访问
        content = msg.content or ""
        tc = msg.tool_calls if hasattr(msg, "tool_calls") else []
        prefix = "  "
        if tc:
            print(f"{prefix}[{i}] 🤖 AI → 工具调用: {[(t['name'], t['args']) for t in tc]}")
        elif role == "human":
            print(f"{prefix}[{i}] 👤 {content[:60]}")
        elif role == "ai":
            print(f"{prefix}[{i}] 🤖 {content[:100]}")
        elif role == "tool":
            print(f"{prefix}[{i}] 📦 {content[:80]}")

## 3. stream_mode="messages" — 逐 token 实时输出

> `"messages"` 模式下，LLM 生成的每个 token 都会以消息块的形式独立 emit。
> 搭配 `msg_chunk.content` 即可实现打字机效果。
>
> **注意**：返回的是 `(msg_chunk, metadata)` 元组，metadata 包含节点来源等信息。

In [ ]:
agent = make_agent()

print("【stream_mode='messages'】— 逐 token 输出")
print("=" * 50)
print("Agent：", end="", flush=True)

for msg_chunk, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "深圳天气怎么样？用一句话回答"}]},
    {"configurable": {"thread_id": "messages-001"}},
    stream_mode="messages",
):
    # metadata 包含节点信息
    node = metadata.get("langgraph_node", "")
    
    if node == "agent":
        # agent 节点：LLM 生成的内容
        content = msg_chunk.content
        if content:
            print(content, end="", flush=True)
        # 工具调用信息
        if msg_chunk.tool_calls:
            for tc in msg_chunk.tool_calls:
                print(f"\n🔧 调用: {tc['name']}(args={tc['args']})", flush=True)
    elif node == "tools":
        # 工具节点：工具返回结果
        content = msg_chunk.content
        if content:
            print(f"\n📦 工具结果: {content[:60]}", flush=True)

print("\n" + "=" * 50)

## 4. stream_mode="messages" + tool_calls 实时展示

> 当 Agent 调用多个工具时，`stream_mode="messages"` 能清晰展示每一步的思考、工具调用、工具结果。

In [ ]:
agent = make_agent()

print("【多工具调用 — stream_mode='messages'】")
print("=" * 50)

for msg_chunk, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "现在几点了？再查一下广州天气，最后算 (15+25)*2"}]},
    {"configurable": {"thread_id": "multi-tools-001"}},
    stream_mode="messages",
):
    node = metadata.get("langgraph_node", "")
    
    if node == "agent":
        content = msg_chunk.content
        if content:
            print(content, end="", flush=True)
        if msg_chunk.tool_calls:
            for tc in msg_chunk.tool_calls:
                rprint(f"\n[yellow]🔧 调用工具: {tc['name']}({tc['args']})[/yellow]")
    elif node == "tools":
        content = msg_chunk.content
        if content:
            rprint(f"[green]📦 工具结果: {content[:80]}[/green]")

## 5. astream() + stream_mode 异步流式

> `.astream()` 是异步版本，支持所有 `stream_mode`，适合 Web 框架。

In [ ]:
import asyncio

agent = make_agent()

async def demo_astream():
    print("【异步流式 astream() + stream_mode='messages'】")
    print("=" * 50)
    print("Agent：", end="", flush=True)
    
    async for msg_chunk, metadata in agent.astream(
        {"messages": [{"role": "user", "content": "现在几点了？"}]},
        {"configurable": {"thread_id": "async-001"}},
        stream_mode="messages",
    ):
        node = metadata.get("langgraph_node", "")
        if node == "agent":
            content = msg_chunk.content
            if content:
                print(content, end="", flush=True)
                await asyncio.sleep(0.02)  # 放慢一点，更直观
        elif node == "tools":
            if msg_chunk.content:
                print(f"\n📦 {msg_chunk.content}", flush=True)
    
    print("\n" + "=" * 50)
    print("✅ 异步流式完成")

await demo_astream()

---

# 第二部分：输出模式

> `agent.invoke()` 返回的字典包含整个 Agent 状态，不同方式获取不同维度的输出。

## 6. 直接输出 — messages[-1]['content']

> 取消息列表的最后一条。如果最后是 ToolMessage，需要用第 8 节的方法过滤。

In [ ]:
agent = make_agent()

result = agent.invoke(
    {"messages": [{"role": "user", "content": "广州天气怎么样？"}]},
    {"configurable": {"thread_id": "mode-001"}},
)

last = result['messages'][-1]
rprint(f"[bold]最后一条消息:[/bold]")
rprint(f"  角色: {last.type}")
rprint(f"  内容: {last.content}")

## 7. 查看完整消息链

> `result['messages']` 包含整个执行过程：用户提问 → AI 思考 → 调工具 → 工具结果 → AI 回答。

In [ ]:
agent = make_agent()

result = agent.invoke(
    {"messages": [{"role": "user", "content": "算一下 (8+12)*5，再查北京天气"}]},
    {"configurable": {"thread_id": "chain-001"}},
)

msgs = result['messages']
rprint(f"[bold]消息链（共 {len(msgs)} 条）：[/bold]")

for i, msg in enumerate(msgs):
    role = msg.type  # Pydantic 对象用属性访问
    content = msg.content or ""
    tc = msg.tool_calls if hasattr(msg, "tool_calls") else []
    icon = {"human": "👤", "ai": "🤖", "tool": "📦"}.get(role, "❓")

    if tc:
        tools_str = ", ".join(f"{t['name']}({t['args']})" for t in tc)
        print(f"  [{i}] {icon} AI 调用: {tools_str}")
    elif role == "tool":
        print(f"  [{i}] {icon} 结果: {str(content)[:60]}")
    else:
        print(f"  [{i}] {icon} {str(content)[:80]}")

print(f"\n✅ 最终答案: {msgs[-1].content[:100]}")

## 8. 过滤 AI 回答（跳过工具消息）

> 当最后一条消息是 ToolMessage 时，`messages[-1]` 不是最终回答。
> 需要反向遍历找到最后一个 AI 的文本回答。

In [ ]:
agent = make_agent()

result = agent.invoke(
    {"messages": [{"role": "user", "content": "北京上海天气都查一下"}]},
    {"configurable": {"thread_id": "filter-001"}},
)

def get_final_answer(messages):
    """从消息列表中提取最后一个 AI 文本回答。"""
    for msg in reversed(messages):
        role = msg.type
        content = msg.content or ""
        tc = msg.tool_calls if hasattr(msg, "tool_calls") else []
        if role == "ai" and content and not tc:
            return content
    return messages[-1].content

final = get_final_answer(result['messages'])
rprint(f"[bold]最终 AI 回答：[/bold]\n{final}")

## 9. response_format 模式：结构化输出

> `create_agent` 的 `response_format` 参数让 Agent 最终输出按 Pydantic 模型结构返回。

In [ ]:
from pydantic import BaseModel, Field

class CityWeatherInfo(BaseModel):
    """城市天气信息。"""
    city: str = Field(description="城市名称")
    temperature: str = Field(description="温度")
    condition: str = Field(description="天气状况")
    advice: str = Field(description="出行建议")

# 携带 response_format 创建 Agent
format_agent = create_agent(
    model="openai:" + os.getenv("MODEL_NAME"),
    tools=tools,
    system_prompt="你是一个天气助手，先查天气再回答。",
    response_format=CityWeatherInfo,
    checkpointer=MemorySaver(),
)

result = format_agent.invoke(
    {"messages": [{"role": "user", "content": "深圳天气怎么样？"}]},
    {"configurable": {"thread_id": "rf-001"}},
)

# Pydantic 对象用属性访问
last_msg = result['messages'][-1]
content = last_msg.content

if isinstance(content, dict):
    weather = CityWeatherInfo.model_validate(content)
    rprint(f"[bold]结构化输出（Pydantic 模型）：[/bold]")
    rprint(f"  城市：{weather.city}")
    rprint(f"  温度：{weather.temperature}")
    rprint(f"  天气：{weather.condition}")
    rprint(f"  建议：{weather.advice}")
else:
    rprint(f"[yellow]content 不是 dict，当前类型: {type(content).__name__}[/yellow]")
    rprint(f"content = {content}")

---

## 10. stream_mode 对比总结

| stream_mode | emit 粒度 | 输出内容 | 适用场景 |
|-------------|----------|---------|---------|
| `"updates"`（默认） | 每节点 | 增量变更（仅变化的节点） | 标准调试、追踪工具调用链 |
| `"values"` | 每节点 | 完整状态快照（所有消息） | 需要每步完整上下文的场景 |
| `"messages"` | 逐 token | `(MessageChunk, metadata)` 元组 | 打字机效果、实时展示、Web UI |

| 输出模式 | 获取方式 | 返回内容 |
|---------|---------|---------|
| 最终文本 | `result['messages'][-1]['content']` | 字符串 |
| 完整消息链 | `result['messages']` | 消息列表 |
| AI 回答过滤 | 反向遍历找 assistant 消息 | 字符串（跳过工具消息） |
| 结构化输出 | `response_format=PydanticModel` | dict / 模型实例 |

In [ ]:
from rich.table import Table

table = Table(title="⭐ create_agent 流式模式选型")
table.add_column("模式", style="cyan", no_wrap=True)
table.add_column("代码示例", style="green")
table.add_column("输出粒度", style="yellow")
table.add_column("典型用途", style="white")

table.add_row(
    "updates",
    'agent.stream(..., stream_mode="updates")',
    "节点级增量",
    "调试、追踪工具链",
)
table.add_row(
    "values",
    'agent.stream(..., stream_mode="values")',
    "节点级完整快照",
    "状态回放、日志审计",
)
table.add_row(
    "messages",
    'agent.stream(..., stream_mode="messages")',
    "逐 token",
    "打字机效果、Web UI",
)

rprint(table)